In [3]:
import shutil
total, used, free = shutil.disk_usage("/")
print(f"Free: {free // (2**30)} GB")

Free: 8 GB


In [4]:
import sys
!{sys.executable} -m pip install open-clip-torch


[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: pip3 install --upgrade pip


In [5]:
import open_clip
model, _, preprocess = open_clip.create_model_and_transforms("ViT-B-32", pretrained="openai")
print("Model loaded!")


open_clip_model.safetensors:   0%|          | 0.00/605M [00:00<?, ?B/s]

/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/open_clip/factory.py:450: UserWarning: QuickGELU mismatch between final model config (quick_gelu=False) and pretrained tag 'openai' (quick_gelu=True).
  warnings.warn(


Model loaded!


In [7]:
import os
fragment_dir = os.path.expanduser("~/Downloads/fragments")
print(os.listdir(fragment_dir))

['fragment_shoes.png', 'fragment_bracelet.png', '.DS_Store', 'fragment_yellowball.png', 'fragment_whitebust.png', 'fragment_yellowbust.png', 'fragment_bluegown.png', 'fragment_yellowdetailing.png', 'fragment_whitehem.png', 'fragment_yellowhem.png', 'fragment_yellowflower.png', 'fragment_armcuff.png', 'fragment_chain.png', 'fragment_bluehem.png', 'fragment_whitewings.png', 'fragment_whitewaist.png']


In [8]:
import os
import torch
import open_clip
from PIL import Image
import json

# Load model
device = "cpu"
model, _, preprocess = open_clip.create_model_and_transforms("ViT-B-32", pretrained="openai")
model.eval()

# Paths
fragment_dir = os.path.expanduser("~/Downloads/fragments")
fragments = [
    os.path.join(fragment_dir, f) for f in os.listdir(fragment_dir)
    if f.startswith("fragment_") and f.endswith(".png")
]

print(f"Found {len(fragments)} fragments")

scraped_dir = os.path.expanduser("~/Downloads/vogue_images")
scraped_images = [
    os.path.join(scraped_dir, f) for f in os.listdir(scraped_dir)
    if f.endswith(".jpg")
]

print(f"Found {len(scraped_images)} scraped images")

def get_embedding(image_path):
    image = preprocess(Image.open(image_path).convert("RGB")).unsqueeze(0)
    with torch.no_grad():
        embedding = model.encode_image(image)
    return embedding / embedding.norm(dim=-1, keepdim=True)

print("Getting embeddings for scraped images...")
scraped_embeddings = []
for path in scraped_images:
    try:
        emb = get_embedding(path)
        scraped_embeddings.append((path, emb))
    except:
        pass

print(f"Loaded {len(scraped_embeddings)} scraped embeddings\n")

results = []

for frag_path in fragments:
    frag_name = os.path.basename(frag_path)
    frag_emb = get_embedding(frag_path)

    best_score = -1
    best_match = None

    for path, emb in scraped_embeddings:
        score = torch.nn.functional.cosine_similarity(frag_emb, emb).item()
        if score > best_score:
            best_score = score
            best_match = path

    if best_match:
        results.append({
            "fragment": frag_name,
            "best_match": os.path.basename(best_match),
            "score": round(best_score, 3)
        })

# Save JSON
with open(os.path.expanduser("~/Downloads/clip_results.json"), "w") as f:
    json.dump(results, f, indent=2)

# Print and save as text list
with open(os.path.expanduser("~/Downloads/clip_matches.txt"), "w") as f:
    for r in results:
        line = f"{r['fragment']} → {r['best_match']} (score: {r['score']})"
        print(line)
        f.write(line + "\n")

print(f"\nDone! Matched {len(results)} fragments")
print("Saved to ~/Downloads/clip_matches.txt")

Found 15 fragments
Found 112 scraped images
Getting embeddings for scraped images...
Loaded 112 scraped embeddings

fragment_shoes.png → vogue_image_10.jpg (score: 0.697)
fragment_bracelet.png → vogue_image_10.jpg (score: 0.72)
fragment_yellowball.png → vogue_image_10.jpg (score: 0.667)
fragment_whitebust.png → vogue_image_10.jpg (score: 0.726)
fragment_yellowbust.png → vogue_image_10.jpg (score: 0.716)
fragment_bluegown.png → vogue_image_10.jpg (score: 0.722)
fragment_yellowdetailing.png → vogue_image_10.jpg (score: 0.704)
fragment_whitehem.png → vogue_image_10.jpg (score: 0.753)
fragment_yellowhem.png → vogue_image_10.jpg (score: 0.701)
fragment_yellowflower.png → vogue_image_25.jpg (score: 0.66)
fragment_armcuff.png → vogue_image_85.jpg (score: 0.74)
fragment_chain.png → vogue_image_10.jpg (score: 0.693)
fragment_bluehem.png → vogue_image_10.jpg (score: 0.738)
fragment_whitewings.png → vogue_image_10.jpg (score: 0.742)
fragment_whitewaist.png → vogue_image_10.jpg (score: 0.685)

Don

In [9]:
import pandas as pd

df = pd.DataFrame(results)

# Clean up column names
df["fragment"] = df["fragment"].str.replace("fragment_", "").str.replace(".png", "")
df["best_match"] = df["best_match"].str.replace("vogue_image_", "Vogue Image ").str.replace(".jpg", "")
df.columns = ["Fragment", "Best Match", "Similarity Score"]
df = df.sort_values("Similarity Score", ascending=False)

# Display as table
df.style.background_gradient(subset=["Similarity Score"], cmap="RdYlGn")

,Fragment,Best Match,Similarity Score
7,whitehem,Vogue Image 10,0.753000
13,whitewings,Vogue Image 10,0.742000
10,armcuff,Vogue Image 85,0.740000
12,bluehem,Vogue Image 10,0.738000
3,whitebust,Vogue Image 10,0.726000
5,bluegown,Vogue Image 10,0.722000
1,bracelet,Vogue Image 10,0.720000
4,yellowbust,Vogue Image 10,0.716000
6,yellowdetailing,Vogue Image 10,0.704000
8,yellowhem,Vogue Image 10,0.701000


In [10]:
import pandas as pd

df = pd.DataFrame(results)

# Clean up column names
df["fragment"] = df["fragment"].str.replace("fragment_", "").str.replace(".png", "")
df["best_match"] = df["best_match"].str.replace("vogue_image_", "Vogue Image ").str.replace(".jpg", "")
df.columns = ["Fragment", "Best Match", "Similarity Score"]

# Filter out vogue image 10
df = df[df["Best Match"] != "Vogue Image 10"]

# Sort by score
df = df.sort_values("Similarity Score", ascending=False)

# Display as colour coded table
df.style.background_gradient(subset=["Similarity Score"], cmap="RdYlGn")

,Fragment,Best Match,Similarity Score
10,armcuff,Vogue Image 85,0.740000
9,yellowflower,Vogue Image 25,0.660000
